# VisionTracker Remote Server — Google Colab Setup

This notebook sets up a self-hosted vision-language model as a FastAPI server for VisionTracker object identification.

## Features
- Free tier compatible — runs on Colab's free T4 GPU
- Private — your images never leave your server
- Flexible models — supports GGUF and safetensors formats (<10GB)
- Accessible anywhere — via ngrok tunnel

## Prerequisites
1. Create a free ngrok account at https://dashboard.ngrok.com/signup
2. Copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
3. (Optional) Set it as a Colab secret: Click 🔑 in left sidebar → Add secret: NGROK_AUTHTOKEN

## Cell 1: Configure Model

Choose your model format and configure the model path/URL.

### Model Format Options:
- GGUF (recommended for <10GB models): Use with llama-cpp-python
- safetensors (for HuggingFace models): Use with transformers

### Upload your own model:
1. Upload GGUF or safetensors files to Colab (files panel on left)
2. Set MODEL_PATH to /content/your-model.gguf or HF repo ID

In [ ]:
# CONFIGURATION — Edit these values

# Model format: "gguf" or "safetensors"
MODEL_FORMAT = "gguf"

# Model path or HuggingFace repo ID
MODEL_PATH = "https://huggingface.co/lmstudio-community/gemma-3-4b-it-GGUF/resolve/main/gemma-3-4b-it-Q4_K_M.gguf"

# For GGUF: filename to save as
MODEL_FILENAME = "gemma-3-4b-it-Q4_K_M.gguf"

# Context window size
N_CTX = 4096

# GPU layers (-1 = use all layers, 0 = CPU only)
N_GPU_LAYERS = -1

print(f"Config: format={MODEL_FORMAT}, model={MODEL_PATH}")

## Cell 2: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok pillow pydantic

if MODEL_FORMAT == "gguf":
    !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python
else:
    !pip install -q transformers accelerate torch

print("Dependencies installed!")

## Cell 3: Configure ngrok

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

try:
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
    print("Loaded NGROK_AUTHTOKEN from Colab secrets")
except Exception:
    NGROK_TOKEN = input("Enter your ngrok authtoken: ").strip()

if not NGROK_TOKEN:
    raise ValueError("ngrok authtoken is required!")

ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok configured!")

## Cell 4: Download/Load Model

In [ ]:
import os

if MODEL_FORMAT == "gguf" and MODEL_PATH.startswith("http"):
    LOCAL_MODEL_PATH = f"/content/{MODEL_FILENAME}"
    if not os.path.exists(LOCAL_MODEL_PATH):
        print(f"Downloading {MODEL_FILENAME}...")
        !wget -q --show-progress {MODEL_PATH} -O {LOCAL_MODEL_PATH}
    else:
        print("Model already exists")
    print(f"Model size: {os.path.getsize(LOCAL_MODEL_PATH) / 1e9:.2f} GB")
else:
    LOCAL_MODEL_PATH = MODEL_PATH
    print(f"Using model: {LOCAL_MODEL_PATH}")

## Cell 5: Load Model

In [ ]:
if MODEL_FORMAT == "gguf":
    from llama_cpp import Llama
    print("Loading GGUF model...")
    llm = Llama(
        model_path=LOCAL_MODEL_PATH,
        n_ctx=N_CTX,
        n_gpu_layers=N_GPU_LAYERS,
        verbose=False
    )
    _ = llm("Say 'ready'", max_tokens=10)
    print("Model loaded and warmed up!")
else:
    from transformers import AutoProcessor, AutoModelForVision2Seq
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None
    )
    if device == "cpu":
        model = model.to("cpu")
    print(f"Model loaded on {device}!")

## Cell 6: Create FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from typing import List, Optional
from PIL import Image
import io, base64, re

app = FastAPI(title="VisionTracker Remote Server", version="2.0.0")

try:
    SERVER_API_KEY = userdata.get('SERVER_API_KEY')
except:
    SERVER_API_KEY = None

class IdentifyRequest(BaseModel):
    annotated_image: str
    track_ids: List[int]

class IdentifyResponse(BaseModel):
    results: List[dict]

def create_prompt(n):
    if n == 1:
        return "Describe the object inside the colored box in ONE sentence (<= 12 words)."
    numbered = chr(10).join([f"{i+1}. <desc>" for i in range(n)])
    return f"Describe each object in colored boxes. One sentence each (<=12 words). Format:\n{numbered}"

def parse_response(text, n, track_ids):
    results = []
    pattern = re.compile(r"^\s*(\d+)[.)]\s*(.+)$")
    numbered = {}
    for line in text.splitlines():
        m = pattern.match(line)
        if m:
            idx = int(m.group(1)) - 1
            if 0 <= idx < n:
                numbered[idx] = m.group(2).strip()
    for i in range(n):
        results.append(numbered.get(i, f"object #{track_ids[i]}"))
    return results

def identify_gguf(image_b64, track_ids):
    n = len(track_ids)
    prompt = create_prompt(n)
    content = [{"type": "text", "text": prompt}]
    content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}})
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],
        max_tokens=100*n, temperature=0.2, stop=["</s>"]
    )
    return parse_response(response["choices"][0]["message"]["content"].strip(), n, track_ids)

def identify_safetensors(image_b64, track_ids):
    import torch
    n = len(track_ids)
    prompt = create_prompt(n)
    img_data = base64.b64decode(image_b64)
    image = Image.open(io.BytesIO(img_data)).convert("RGB")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(images=image, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100*n, temperature=0.2, do_sample=True)
    raw_text = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return parse_response(raw_text, n, track_ids)

@app.get("/health")
async def health():
    return {"status": "healthy", "format": MODEL_FORMAT, "model": str(MODEL_PATH)}

@app.post("/identify")
async def identify(req: IdentifyRequest, auth: Optional[str] = Header(None)):
    if SERVER_API_KEY and auth != f"Bearer {SERVER_API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid API key")
    if not req.track_ids:
        return {"results": []}
    if MODEL_FORMAT == "gguf":
        descs = identify_gguf(req.annotated_image, req.track_ids)
    else:
        descs = identify_safetensors(req.annotated_image, req.track_ids)
    return {"results": [{"track_id": tid, "description": d} for tid, d in zip(req.track_ids, descs)]}

print("FastAPI app created!")

## Cell 7: Start Server

In [ ]:
import nest_asyncio, uvicorn
nest_asyncio.apply()
public_url = ngrok.connect(8000, "http")
print("="*60)
print("SERVER IS RUNNING!")
print("="*60)
print(f"Public URL: {public_url}")
print(f"Health: {public_url}/health")
print(f"Identify: {public_url}/identify")
print("="*60)
print(f"\nUsage: python main.py --remote-url {public_url}")
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")